# Kaggle Launcher — E2VID Reconstruction + YOLO Training
**Last updated: 2026-06-07 00:00  v10**

This notebook is a thin wrapper that runs the two Python scripts on Kaggle GPU.
It is a minimal adaptation of `colab_launcher.ipynb` — the scripts themselves are unchanged.

**Before running:**
1. Add the `fred-events-ami` dataset to this notebook (contains events.zip, coordinates.txt, scripts)
2. Set Runtime → Accelerator → **GPU T4 x2** (or P100)
3. Edit the **Configuration** cell if needed (dataset slug, RESUME flag)
4. Run all cells top to bottom

**Output:** saved automatically to Kaggle output → download `yolo_e2vid.pt` and copy to `services/e2vid/weights/`.

**Session limit:** Kaggle allows up to 9 hours — enough for a full reconstruction + training run without needing resume.

## 1 · Configuration — edit this cell

In [ ]:
from pathlib import Path
import datetime

# ── Kaggle dataset paths ───────────────────────────────────────────────────────
# Two datasets: large data (events + coordinates) and small scripts.
# Update scripts only: bash scripts/sync_scripts_to_kaggle.sh  (~30 KB, seconds)
# Update data only:    bash scripts/sync_to_kaggle.sh           (~2.1 GB, ~7 min)
AMI_INPUT   = Path('/kaggle/input/datasets/gennepy/fred-events-ami')   # events + coordinates
SCRIPTS_INPUT = Path('/kaggle/input/datasets/gennepy/fred-scripts-ami') # reconstruct.py, train_yolo.py
AMI_WORK    = Path('/kaggle/working')                                    # writable output

# ── Resume a previous run that was interrupted? ───────────────────────────────
RESUME = False

# ── Sequences ─────────────────────────────────────────────────────────────────
SEQUENCES     = ['sequence_84', 'sequence_85', 'sequence_201', 'sequence_127']
VAL_SEQUENCES = ['sequence_127']

# ── Reconstruction parameters ─────────────────────────────────────────────────
START_S          = 5.0
EVENTS_PER_PIXEL = 0.01
SMOKE_EVENTS     = None   # e.g. 100_000 for a quick smoke test (~10 frames)

# ── Training parameters ───────────────────────────────────────────────────────
EPOCHS = 100
BATCH  = 16

# ── Derived paths ─────────────────────────────────────────────────────────────
SCRIPTS_DIR = SCRIPTS_INPUT
EVENTS_ROOT = AMI_INPUT / 'data' / 'processed'
RAW_ROOT    = AMI_INPUT / 'data' / 'raw'
RECON_ROOT  = AMI_WORK  / 'data' / 'processed'
WEIGHTS_OUT = AMI_WORK  / 'yolo_e2vid.pt'
RUNS_DIR    = AMI_WORK  / 'yolo_runs'
LOG_FILE    = AMI_WORK  / 'logs' / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
DATASET_DIR = AMI_WORK  / 'yolo_e2vid'

WORK_DIR = AMI_WORK / 'work'
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

train_seqs = [s for s in SEQUENCES if s not in VAL_SEQUENCES]
print('Events input  :', AMI_INPUT)
print('Scripts input :', SCRIPTS_INPUT)
print('Work root     :', AMI_WORK)
print('Train seqs    :', train_seqs)
print('Val seqs      :', VAL_SEQUENCES)
print('Resume        :', RESUME)
print('Smoke events  :', SMOKE_EVENTS or 'full run')
print('Epochs        :', EPOCHS)

## 2 · Install dependencies

In [ ]:
!pip install -q h5py ultralytics==8.4.54 imageio scikit-image pandas matplotlib
import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Go to Settings → Accelerator → GPU T4 x2, then restart.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}  '
      f'({torch.cuda.get_device_properties(0).total_memory // 1024**2} MB)')

## 3 · Helpers

In [ ]:
import os, subprocess, sys, datetime, shutil

# Copy scripts from input dataset to local working dir
LOCAL_SCRIPTS = Path('/kaggle/working/scripts')
LOCAL_SCRIPTS.mkdir(exist_ok=True)
for script in ['reconstruct.py', 'train_yolo.py']:
    shutil.copy(SCRIPTS_DIR / script, LOCAL_SCRIPTS / script)
print(f'Scripts copied to {LOCAL_SCRIPTS}')

def run_streaming(cmd):
    """Run a command, stream output to notebook and append to log file."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(LOG_FILE, 'a') as lf:
        for line in process.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    process.wait()
    return process.returncode

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log('Helpers ready.')

## 4 · Cleanup — remove stale output before each run

In [ ]:
import shutil

if RESUME:
    print('RESUME=True — skipping cleanup, keeping existing outputs.')
else:
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if recon_dir.exists():
            shutil.rmtree(recon_dir)
            print(f'Cleaned: {recon_dir}')
        else:
            print(f'Nothing to clean: {recon_dir}')

    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    print(f'Cleaned: {DATASET_DIR}')

    shutil.rmtree(WORK_DIR / 'rpg_e2vid', ignore_errors=True)
    print(f'Cleaned: {WORK_DIR / "rpg_e2vid"}')

## 5 · Reconstruct e2vid frames

In [ ]:
log('=== Reconstruction started ===')

for seq in SEQUENCES:
    zip_path = EVENTS_ROOT / seq / 'events.zip'    # read from input dataset
    out_dir  = RECON_ROOT  / seq / 'reconstruction_e2vid'  # write to working

    if out_dir.exists() and (any(out_dir.glob('frame_*.png')) or any(out_dir.glob('frame_*.jpg'))):
        log(f'{seq}: frames already exist — skipping reconstruction')
        continue

    log(f'=== Reconstructing {seq} ===')
    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'reconstruct.py'),
        '--zip_path',         str(zip_path),
        '--out_dir',          str(out_dir),
        '--work_dir',         str(WORK_DIR),
        '--events_per_pixel', str(EVENTS_PER_PIXEL),
        '--compress_jpeg',    # convert PNGs → JPEG after each sequence (~5x smaller)
    ]
    if SMOKE_EVENTS:
        cmd += ['--max_events', str(SMOKE_EVENTS)]

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: reconstruct.py failed for {seq} (exit code {rc})')
        raise RuntimeError(f'reconstruct.py failed for {seq} (exit code {rc})')

log('=== Reconstruction done ===')

## 6 · Train YOLO

In [ ]:
log('=== Training started ===')

# On resume, the checkpoint stores the yaml path from the previous session.
# Pre-write the yaml at that same path so ultralytics finds it immediately.
resume_yaml = DATASET_DIR / 'dataset.yaml'
resume_yaml.parent.mkdir(parents=True, exist_ok=True)
resume_yaml.write_text(f"""path: {DATASET_DIR}
train: train.txt
val:   val.txt

nc: 1
names:
  0: drone
""")
log(f'dataset.yaml pre-written → {DATASET_DIR}')

cmd = [
    sys.executable, str(LOCAL_SCRIPTS / 'train_yolo.py'),
    '--sequences',  *SEQUENCES,
    '--raw_root',   str(RAW_ROOT),
    '--recon_root', str(RECON_ROOT),
    '--out_dir',    str(DATASET_DIR),
    '--runs_dir',   str(RUNS_DIR),
    '--weights',    str(WEIGHTS_OUT),
    '--epochs',     str(EPOCHS),
    '--batch',      str(BATCH),
]

if VAL_SEQUENCES:
    cmd += ['--val_sequences', *VAL_SEQUENCES]

if RESUME:
    cmd += ['--resume']

rc = run_streaming(cmd)
if rc != 0:
    log(f'ERROR: train_yolo.py failed (exit code {rc})')
    raise RuntimeError('train_yolo.py failed')

log(f'=== Training done — weights at {WEIGHTS_OUT} ===')
print('Download yolo_e2vid.pt from Kaggle output and copy to services/e2vid/weights/')

## 7 · Zip frames for download

Bundles all reconstructed JPEG frames into a single zip — much faster to download
than 40k individual files via the Kaggle API.


In [ ]:
# ── Zip reconstructed frames for fast single-file download ───────────────────
import zipfile, os
from pathlib import Path

WORKING = Path('/kaggle/working')
zip_path = WORKING / 'frames_e2vid.zip'

log('=== Zipping reconstructed frames ===')
frame_count = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for seq in SEQUENCES:
        recon_dir = WORKING / 'data' / 'processed' / seq / 'reconstruction_e2vid'
        ts_file = recon_dir / 'timestamps.txt'
        if ts_file.exists():
            zf.write(ts_file, ts_file.relative_to(WORKING))
        for frame in sorted(recon_dir.glob('frame_*.jpg')):
            zf.write(frame, frame.relative_to(WORKING))
            frame_count += 1

size_mb = zip_path.stat().st_size / 1e6
log(f'frames_e2vid.zip: {frame_count} frames, {size_mb:.0f} MB → {zip_path}')


## 8 · Download results

After the notebook finishes, Kaggle saves `/kaggle/working/` as the run output.

**Weights + KPIs + logs (fast — ~50 MB):**
```bash
bash scripts/sync_from_kaggle.sh
```

**Reconstructed frames (~2.5 GB, single zip):**
```bash
bash scripts/sync_from_kaggle.sh --frames-zip
```

Then rebuild the service:
```bash
docker compose build e2vid && docker compose up -d
```
